# Preparing inputs for a profiling simulation (pt. ii: child simulation)
Here we will prepare the inputs for a simulation to be used for scaling analysis and profiling, including all features supported by \[C\]Worthy's fork of UCLA-ROMS. In order to run the profiling simulation, we first need to run a pair of simulations in order to generate inputs: a parent, from which boundary conditions will be generated for a child, and then the child, which generates additional inputs for a second run of the parent.

Note that the simulation setup is effectively the same as the nested CDR tutorial in the ROMS documentation, but with the domain shifted southward such that there is no land.

In [ ]:
%matplotlib inline

In [ ]:
from pathlib import Path

import xarray as xr
import datetime as dt
import roms_tools as rt
import matplotlib.pyplot as plt

After setting this cell according to your own source data paths, the rest of the notebook should run:

In [ ]:
# ROMS tools source data:
rtd=Path("~/Code/roms_tools_datasets")

topo_path = rtd/"SRTM15_V2.5.nc" # Topography (SRTM15)
era5_path = [rtd/"ERA5_2012-01.nc", rtd/"ERA5_2012-02.nc"] # Surface forcing (ERA5)
bgc_path = rtd/"BGC/BGCdataset.nc" # BGC tracers for initial and boundary conditions (CWorthy unified BGC dataset)
bgc_surf_path = rtd/"BGC/BGCdataset.nc" # BGC surface forcing (CWorthy unified BGC dataset)
glorys_paths = [rtd/f"GLORYS/mercatorglorys12v1_gl12_mean_2012{m:02}{d:02}.nc" for m in range(1,3) for d in range(1,32)][:-2] # Interior state for initial and boundary conditions (GLORYS)
tpxo_path = rtd/"TPXO10.v2/" # Tidal forcing (TPXO)

In [ ]:
# Set directory in which to save files
save_dir=f"{os.environ.get("ROMS_ROOT")}/profiling/input_data"

## Simulation timing
We will run the simulation for the month of February, using the restart and boundary data generated by the parent after its first month.

In [ ]:
forcing_start_time = dt.datetime(2012,2,1,0,0,0)
forcing_end_time = dt.datetime(2012,3,1,0,0,0)
ini_time = dt.datetime(2012,2,1,0,0,0)

## Simulation domain
We saved the grid when preparing the parent inputs, so we can import it here:

In [ ]:
child_grid = rt.Grid.from_file(save_dir+"/child_grid.nc")
child_grid.plot()

The grid-related parameters are set in `roms.in`, as described on the previous page of the tutorial.

## Pre-processing boundary and initial condition data from the parent simulation
### Boundary forcing
During the parent simulation, we used the nesting information generated with `roms-tools` to output `_ext` files, which we can now process into boundary forcing files for the child domain. The tool for this is `extract_data_join`, included with `roms`:

In [ ]:
%%bash
cd $ROMS_ROOT/profiling/input_data
# We want to loop over the filename stems, so we'll use the `.0.nc` files as a template
for F in $ROMS_ROOT/profiling/prepare_inputs/i_parent/output/parent_ext.??????????????.0.nc;do

    # Get filename stem, e.g. 
    # output/roms_ext.20120101120000.0.nc -> output/roms_ext.20120101120000
    filename_stem=${F/.0.nc}  

    # wildcard to join all subdomains with this stem
    extract_data_join ${filename_stem}.?.nc
done

In [ ]:
%%bash
cd $ROMS_ROOT/profiling/input_data/
ncrcat child_bry.??????????????.nc  -o ../input_data/child_boundary_forcing.nc
rm ../input_data/ocean.??????????????.nc
rm ../input_data/child_bry.??????????????.nc
ls ../input_data

### Initial conditions
`roms-tools` allows us to create the initial conditions for the child simulation using a restart file and grid file from the parent simulation.

In [ ]:
restart_file=f"../i_parent/output/parent_rst.20120201000000.nc"
parent_grid = rt.Grid.from_file(save_dir+"/parent_grid.nc")

initial_conditions_from_parent = rt.InitialConditions(
    grid=child_grid,
    ini_time=dt.datetime(2012,2,1,0,0,0),
    source={"name": "ROMS", "grid": parent_grid, "path": restart_file},
    use_dask=True,
    bgc_source={
        "name": "ROMS",
        "grid": parent_grid,
        "path": restart_file,
    },
)
child_initial_conditions_path = initial_conditions_from_parent.save(save_dir+"/child_initial_conditions.nc")

More information on this process can be found in the [`roms-tools` documentation](https://roms-tools.readthedocs.io/en/latest/initial_conditions.html#Initializing-from-ROMS-Restart-Files).

## Creating other input data
We make the remaining input data using `roms-tools` as we did on the previous page of the tutorial:
### Surface forcing

In [ ]:
child_surface_forcing = rt.SurfaceForcing(
    grid = child_grid,
    start_time = forcing_start_time,
    end_time= forcing_end_time,
    type= "physics",
    source={"name": "ERA5", "path": era5_path},
    use_dask=True,
)

child_surface_forcing_path = child_surface_forcing.save(save_dir+"/child_surface_forcing.nc",group=False)

### BGC Surface Forcing

In [ ]:
child_bgc_surface_forcing = rt.SurfaceForcing(
    grid=child_grid,
    start_time=forcing_start_time,
    end_time=forcing_end_time,
    source={"name": "UNIFIED", "path": bgc_surf_path, "climatology": True},
    type="bgc",
    use_dask=True,
)
child_bgc_surface_forcing_path = child_bgc_surface_forcing.save(save_dir+"/child_bgc_surface_forcing.nc",group=False)

## Tidal Forcing

In [ ]:
tpxo_dict = {
    "grid": tpxo_path / "grid_tpxo10v2.nc",
    "h": tpxo_path / "h_tpxo10.v2.nc",
    "u": tpxo_path / "u_tpxo10.v2.nc",
}

child_tidal_forcing = rt.TidalForcing(
    grid=child_grid,
    source={"name": "TPXO", "path": tpxo_dict},
    use_dask=True
)
child_tidal_forcing.plot("ssh_Re", ntides=0)

In [ ]:
child_tidal_forcing_path = child_tidal_forcing.save(save_dir+"/child_tides.nc")

## CDR Forcing
We now prepare an Ocean Alkalinity Enhancement perturbation using `roms-tools`. `roms-tools` allows us to add a time-varying flux to a tracer (here alkalinity) with a Gaussian spatial distribution in the horizontal and vertical, specified by the parameters `hsc` and `vsc`:

In [ ]:
constant_oae = rt.TracerPerturbation(
    name="oae",
    lat=52.0,  # degree N
    lon=-26,  # degree E
    depth=100,  # m
    hsc=100000,  # Gaussian horizontal scale, m
    vsc=50,  # Gaussian vertical scale, m
    tracer_fluxes={"ALK": 2 * 10**6},  # meq/s
)
child_cdr_forcing = rt.CDRForcing(
    grid=child_grid,
    start_time=forcing_start_time,
    end_time=forcing_end_time,
    releases=[
        constant_oae,
    ],
)

In [ ]:
child_cdr_forcing.plot_distribution(release_name="oae")

In [ ]:
child_cdr_forcing.save(save_dir+"/child_cdr_forcing.nc")

_(Cell output hidden due to length)_

## Post-processing (only run after completing this simulation)

### Join `uscl` files:

In [ ]:
%%bash
# Join upscaling files:
cd output
# We want to loop over the filename stems, so we'll use the `.0.nc` files as a template
for F in *.0.nc;do

    # Get filename stem, e.g. 
    # output/roms_uscl.20120201000000.0.nc -> output/roms_uscl.20120201000000
    filename_stem=${F/.0.nc}  

    # wildcard to join all subdomains with this stem
    ncjoin ${filename_stem}.?.nc
done

### Convert `uscl` files to CDR forcing for profiling run

In [ ]:
%%bash
# Convert `_uscl` to cdr forcing files for parent run
uscl_to_cdr $ROMS_ROOT/profiling/prepare_inputs/ii_child/output/iceland_child_uscl.??????????????.nc -o \
$ROMS_ROOT/profiling/input_data/cdr_release_profiles_from_child.nc